## Environment and Data Loading

Locate `assignment_1/data` and load all fields as strings to preserve raw formats and special codes. These are input types, not final schema.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 80)
pd.set_option("display.max_colwidth", 90)

FILES = {
    "attributes": "features_attributes.csv",
    "financials": "features_financials.csv",
    "clickstream": "feature_clickstream.csv",
    "loans": "lms_loan_daily.csv",
}

def locate_assignment():
    for base in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (base, base / "assignment_1"):
            if all((candidate / "data" / name).is_file() for name in FILES.values()):
                return candidate.resolve()
    raise FileNotFoundError(
        "Cannot find files."
    )

ASSIGNMENT_DIR = locate_assignment()
tables = {
    name: pd.read_csv(ASSIGNMENT_DIR / "data" / filename, dtype="string", keep_default_na=False)
    for name, filename in FILES.items()
}


## Dataset Overview

Inspect table sizes, customer counts, columns, and sample values. Names and SSNs are excluded from previews and planned model features.

In [ ]:
overview = pd.DataFrame([
    {
        "table": name, "rows": len(df), "columns": len(df.columns),
        "customers": df["Customer_ID"].nunique(),
        "exact_duplicate_rows": int(df.duplicated().sum()),
        "memory_mb": round(df.memory_usage(deep=True).sum() / 1024**2, 2),
    }
    for name, df in tables.items()
]).set_index("table")
display(overview)

for name, df in tables.items():
    print("\nTABLE:", name)
    display(pd.DataFrame([
        {
            "column": col, "loaded_dtype": str(df[col].dtype),
            "distinct_raw_values": df[col].nunique(dropna=False),
            "raw_examples": "[not displayed]" if col in {"Name", "SSN"}
                            else repr(df[col].drop_duplicates().head(3).tolist()),
        }
        for col in df.columns
    ]))

,rows,columns,customers,exact_duplicate_rows,memory_mb
table,,,,,
attributes,12500,6,12500,0,4.11
financials,12500,22,12500,0,15.81
clickstream,215376,22,8974,0,236.38
loans,137500,11,12500,0,81.08



TABLE: attributes


,column,loaded_dtype,distinct_raw_values,raw_examples
0,Customer_ID,string,12500,"['CUS_0x1000', 'CUS_0x1009', 'CUS_0x100b']"
1,Name,string,10139,[not displayed]
2,Age,string,301,"['18', '26', '19']"
3,SSN,string,11798,[not displayed]
4,Occupation,string,16,"['Lawyer', 'Mechanic', 'Media_Manager']"
5,snapshot_date,string,25,"['2023-05-01', '2025-01-01', '2024-03-01']"



TABLE: financials


,column,loaded_dtype,distinct_raw_values,raw_examples
0,Customer_ID,string,12500,"['CUS_0x1000', 'CUS_0x1009', 'CUS_0x100b']"
1,Annual_Income,string,12492,"['30625.94', '52312.68_', '113781.38999999998']"
2,Monthly_Inhand_Salary,string,12490,"['2706.1616666666664', '4250.39', '9549.7825']"
3,Num_Bank_Accounts,string,174,"['6', '1', '3']"
4,Num_Credit_Card,string,284,"['5', '4', '3']"
5,Interest_Rate,string,300,"['27', '17', '1']"
6,Num_of_Loan,string,87,"['2', '4', '0']"
7,Type_of_Loan,string,6261,"['Credit-Builder Loan, and Home Equity Loan', 'Not Specified, Home Equity Loan, Credit..."
8,Delay_from_due_date,string,73,"['57', '5', '14']"
9,Num_of_Delayed_Payment,string,168,"['26', '18', '8']"



TABLE: clickstream


,column,loaded_dtype,distinct_raw_values,raw_examples
0,fe_1,string,763,"['63', '-108', '-13']"
1,fe_2,string,760,"['118', '182', '8']"
2,fe_3,string,766,"['80', '123', '87']"
3,fe_4,string,764,"['121', '4', '166']"
4,fe_5,string,764,"['55', '-56', '214']"
5,fe_6,string,755,"['193', '27', '-98']"
6,fe_7,string,778,"['111', '25', '215']"
7,fe_8,string,768,"['112', '-6', '152']"
8,fe_9,string,756,"['-101', '284', '129']"
9,fe_10,string,773,"['83', '222', '139']"



TABLE: loans


,column,loaded_dtype,distinct_raw_values,raw_examples
0,loan_id,string,12500,"['CUS_0x1000_2023_05_01', 'CUS_0x1009_2025_01_01', 'CUS_0x100b_2024_03_01']"
1,Customer_ID,string,12500,"['CUS_0x1000', 'CUS_0x1009', 'CUS_0x100b']"
2,loan_start_date,string,25,"['2023-05-01', '2025-01-01', '2024-03-01']"
3,tenure,string,1,['10']
4,installment_num,string,11,"['0', '1', '2']"
5,loan_amt,string,1,['10000']
6,due_amt,string,2,"['0.0', '1000.0']"
7,paid_amt,string,5,"['0.0', '1000.0', '2000.0']"
8,overdue_amt,string,11,"['0.0', '1000.0', '2000.0']"
9,balance,string,11,"['10000.0', '9000.0', '8000.0']"


## Missing Values and Special Codes

Review fields relevant to feature design. Blank `Type_of_Loan` values require comparison with `Num_of_Loan` in Section 7.

Unknown categories may be represented as `Unknown` regardless of their frequency.

In [ ]:
def missing_candidates(series):
    text = series.astype("string").str.strip()
    return (
        text.isna() | text.eq("") |
        text.str.lower().isin(["nan", "null", "none", "n/a", "na"]) |
        text.str.fullmatch(r"_+", na=False)
    ).fillna(True)

MISSING_REVIEW_PCT = 20  # Review priority only; no automatic imputation or column removal.
focus_fields = [
    ("attributes", "Occupation"),
    ("financials", "Type_of_Loan"),
    ("financials", "Changed_Credit_Limit"),
    ("financials", "Credit_Mix"),
    ("financials", "Payment_of_Min_Amount"),
    ("financials", "Payment_Behaviour"),
]
missing_rows = []
for name, col in focus_fields:
    raw = tables[name][col].str.strip()
    missing = missing_candidates(raw)
    # Count observed special codes separately from missing values.
    special = pd.Series(False, index=raw.index)
    if col == "Payment_Behaviour":
        special = raw.eq("!@9#%8")
    elif col == "Type_of_Loan":
        special = raw.str.lower().eq("not specified")
    marked = missing | special
    pct = round(marked.mean() * 100, 2)
    missing_rows.append({
        "table": name, "column": col,
        "missing_candidate_rows": int(missing.sum()),
        "special_code_rows": int(special.sum()),
        "review_pct": pct,
        "priority_review": pct >= MISSING_REVIEW_PCT,
        "examples": repr(raw[marked].value_counts().head(3).to_dict()),
    })
missing_report = pd.DataFrame(missing_rows)
display(missing_report)
# Keep NM separate; its meaning is unconfirmed. See Section 7.

,table,column,missing_candidate_rows,special_code_rows,review_pct,priority_review,examples
0,attributes,Occupation,880,0,7.04,False,{'_______': 880}
1,financials,Type_of_Loan,1426,176,12.82,False,"{'': 1426, 'Not Specified': 176}"
2,financials,Changed_Credit_Limit,254,0,2.03,False,{'_': 254}
3,financials,Credit_Mix,2611,0,20.89,True,{'_': 2611}
4,financials,Payment_of_Min_Amount,0,0,0.00,False,{}
5,financials,Payment_Behaviour,0,998,7.98,False,{'!@9#%8': 998}


## Date Coverage and Monthly Volumes

Check date parsing, coverage, month-start alignment, and gaps between the first and last observed months. Monthly totals count records, not defaults.

In [ ]:
date_rows = []
monthly_counts = {}
for name, df in tables.items():
    for col in [c for c in ["snapshot_date", "loan_start_date"] if c in df.columns]:
        parsed = pd.to_datetime(df[col], format="%Y-%m-%d", errors="coerce")
        valid = parsed.dropna()
        observed = pd.PeriodIndex(valid.dt.to_period("M").unique()).sort_values()
        expected = pd.period_range(observed.min(), observed.max(), freq="M") if len(observed) else observed
        date_rows.append({
            "table": name, "column": col,
            "invalid_or_missing": int(parsed.isna().sum()),
            "first_date": valid.min(), "last_date": valid.max(),
            "distinct_dates": valid.nunique(),
            "not_month_start": int((valid.dt.day != 1).sum()),
            "missing_months_between_endpoints": ", ".join(map(str, expected.difference(observed))),
        })
        if col == "snapshot_date":
            monthly_counts[name] = parsed.dt.to_period("M").value_counts().sort_index()

date_report = pd.DataFrame(date_rows)
display(date_report)
monthly_report = pd.concat(monthly_counts, axis=1).fillna(0).astype(int).sort_index()
monthly_report.index.name = "snapshot_month"
display(monthly_report)

,table,column,invalid_or_missing,first_date,last_date,distinct_dates,not_month_start,missing_months_between_endpoints
0,attributes,snapshot_date,0,2023-01-01,2025-01-01,25,0,
1,financials,snapshot_date,0,2023-01-01,2025-01-01,25,0,
2,clickstream,snapshot_date,0,2023-01-01,2024-12-01,24,0,
3,loans,snapshot_date,0,2023-01-01,2025-11-01,35,0,
4,loans,loan_start_date,0,2023-01-01,2025-01-01,25,0,


,attributes,financials,clickstream,loans
snapshot_month,,,,
2023-01,530,530,8974,530
2023-02,501,501,8974,1031
2023-03,506,506,8974,1537
2023-04,510,510,8974,2047
2023-05,521,521,8974,2568
2023-06,517,517,8974,3085
2023-07,471,471,8974,3556
2023-08,481,481,8974,4037
2023-09,454,454,8974,4491


## Keys, Duplicates, and Record Granularity

Check `Customer_ID + snapshot_date` for feature tables and `loan_id + snapshot_date` for loan records. Distinguish exact duplicates from conflicting records with the same key.

Multiple loan snapshots per customer do not indicate multiple loans.

In [ ]:
key_rows = []
for name, df in tables.items():
    keys = ["loan_id", "snapshot_date"] if name == "loans" else ["Customer_ID", "snapshot_date"]
    key_counts = df.groupby(keys, dropna=False).size()
    distinct_rows = df.drop_duplicates()
    conflicts = distinct_rows.groupby(keys, dropna=False).size()
    key_rows.append({
        "table": name, "candidate_key": " + ".join(keys),
        "rows_with_missing_key": int(pd.concat([missing_candidates(df[k]) for k in keys], axis=1).any(axis=1).sum()),
        "duplicate_key_excess_rows": int(df.duplicated(keys).sum()),
        "duplicate_key_groups": int((key_counts > 1).sum()),
        "conflicting_key_groups": int((conflicts > 1).sum()),
    })
key_report = pd.DataFrame(key_rows)
display(key_report)

customer_grain = pd.DataFrame({
    name: df.groupby("Customer_ID").size().describe()
    for name, df in tables.items()
}).T
print("Records per customer:")
display(customer_grain)

loan_id_counts = tables["loans"].groupby("Customer_ID")["loan_id"].nunique()
print("Distinct loans per customer:")
display(loan_id_counts.value_counts().sort_index().rename_axis("loans_per_customer").to_frame("customers"))

,table,candidate_key,rows_with_missing_key,duplicate_key_excess_rows,duplicate_key_groups,conflicting_key_groups
0,attributes,Customer_ID + snapshot_date,0,0,0,0
1,financials,Customer_ID + snapshot_date,0,0,0,0
2,clickstream,Customer_ID + snapshot_date,0,0,0,0
3,loans,loan_id + snapshot_date,0,0,0,0


Records per customer:


,count,mean,std,min,25%,50%,75%,max
attributes,12500.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0
financials,12500.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0
clickstream,8974.0,24.0,0.0,24.0,24.0,24.0,24.0,24.0
loans,12500.0,11.0,0.0,11.0,11.0,11.0,11.0,11.0


Distinct loans per customer:


,customers
loans_per_customer,
1,12500


## Numeric Formats and Distributions

Separate format issues from numeric distributions. Remove leading and trailing underscores in copies; internal underscores and raw inputs remain unchanged.

Successful parsing does not establish validity. Codes such as `__10000__` and extreme negative balances require separate review. Retain negative clickstream values because their business meaning is undocumented.

In [ ]:
NUMERIC_COLUMNS = {
    "attributes": ["Age"],
    "financials": [
        "Annual_Income", "Monthly_Inhand_Salary", "Num_Bank_Accounts", "Num_Credit_Card",
        "Interest_Rate", "Num_of_Loan", "Delay_from_due_date", "Num_of_Delayed_Payment",
        "Changed_Credit_Limit", "Num_Credit_Inquiries", "Outstanding_Debt",
        "Credit_Utilization_Ratio", "Total_EMI_per_month", "Amount_invested_monthly",
        "Monthly_Balance",
    ],
    "clickstream": [f"fe_{i}" for i in range(1, 21)],
    "loans": ["tenure", "installment_num", "loan_amt", "due_amt", "paid_amt", "overdue_amt", "balance"],
}
numeric_views = {}
format_rows = []

# Reuse one numeric diagnostic copy per source.
for name, columns in NUMERIC_COLUMNS.items():
    numeric_views[name] = pd.DataFrame(index=tables[name].index)
    for col in columns:
        raw = tables[name][col].str.strip()
        missing = missing_candidates(raw)
        direct = pd.to_numeric(raw.mask(missing), errors="coerce")
        preview = pd.to_numeric(raw.mask(missing).str.strip("_"), errors="coerce")
        failures = direct.isna() & ~missing
        non_finite = preview.notna() & ~np.isfinite(preview)
        numeric_views[name][col] = preview.mask(non_finite)
        format_rows.append({
            "table": name, "column": col,
            "missing_candidates": int(missing.sum()),
            "direct_parse_failures": int(failures.sum()),
            "recovered_by_edge_underscore_removal": int((failures & preview.notna()).sum()),
            "remaining_parse_failures": int((preview.isna() & ~missing).sum()),
            "non_finite": int(non_finite.sum()),
            "examples": repr(raw[failures | non_finite].value_counts().head(3).to_dict()),
        })

numeric_report = pd.DataFrame(format_rows)
issue_columns = ["missing_candidates", "direct_parse_failures", "non_finite"]
format_issues = numeric_report[numeric_report[issue_columns].gt(0).any(axis=1)]
print("A. Numeric fields with missing values or format issues")
display(format_issues)

print("B. Numeric distributions before validity checks")
for name in ["attributes", "financials", "loans"]:
    values = numeric_views[name]
    summary = values.describe(percentiles=[.5, .99]).T[["min", "50%", "99%", "max"]]
    summary = summary.rename(columns={"50%": "median", "99%": "p99"})
    summary["negative_count"] = values.lt(0).sum()
    print(name)
    display(summary)

print("C. Clickstream distributions; negative values retained")
click_values = numeric_views["clickstream"]
click_summary = pd.DataFrame({
    "min": click_values.min(), "median": click_values.median(), "max": click_values.max(),
    "negative_pct": (click_values.lt(0).mean() * 100).round(2),
})
display(click_summary)

A. Numeric fields with missing values or format issues


,table,column,missing_candidates,direct_parse_failures,recovered_by_edge_underscore_removal,remaining_parse_failures,non_finite,examples
0,attributes,Age,0,637,637,0,0,"{'32_': 25, '39_': 25, '21_': 23}"
1,financials,Annual_Income,0,859,859,0,0,"{'52312.68_': 1, '14184.935_': 1, '32370.6_': 1}"
6,financials,Num_of_Loan,0,623,623,0,0,"{'2_': 110, '4_': 105, '3_': 79}"
8,financials,Num_of_Delayed_Payment,0,374,374,0,0,"{'18_': 25, '19_': 24, '20_': 23}"
9,financials,Changed_Credit_Limit,254,0,0,0,0,{}
11,financials,Outstanding_Debt,0,139,139,0,0,"{'2699.17_': 1, '2817.77_': 1, '446.31_': 1}"
14,financials,Amount_invested_monthly,0,558,558,0,0,{'__10000__': 558}
15,financials,Monthly_Balance,0,1,1,0,0,{'__-333333333333333333333333333__': 1}


B. Numeric distributions before validity checks
attributes


,min,median,p99,max,negative_count
Age,-500.0,33.0,3920.64,8678.0,104


financials


,min,median,p99,max,negative_count
Annual_Income,7005.93,37572.375,178815.7132,23834698.0,0
Monthly_Inhand_Salary,303.645417,3087.595,13789.2362,15204.633333,0
Num_Bank_Accounts,-1.0,6.0,511.01,1756.0,4
Num_Credit_Card,0.0,5.0,878.01,1499.0,0
Interest_Rate,1.0,14.0,2865.01,5789.0,0
Num_of_Loan,-100.0,3.0,9.0,1495.0,500
Delay_from_due_date,-5.0,18.0,61.0,67.0,88
Num_of_Delayed_Payment,-3.0,14.0,27.0,4293.0,84
Changed_Credit_Limit,-6.49,9.41,28.7955,36.97,204
Num_Credit_Inquiries,0.0,6.0,946.53,2554.0,0


loans


,min,median,p99,max,negative_count
tenure,10.0,10.0,10.0,10.0,0
installment_num,0.0,5.0,10.0,10.0,0
loan_amt,10000.0,10000.0,10000.0,10000.0,0
due_amt,0.0,1000.0,1000.0,1000.0,0
paid_amt,0.0,1000.0,2000.0,4000.0,0
overdue_amt,0.0,0.0,8000.0,10000.0,0
balance,0.0,7000.0,10000.0,10000.0,0


C. Clickstream distributions; negative values retained


,min,median,max,negative_pct
fe_1,-378,102.0,541,15.4
fe_2,-356,103.0,560,14.94
fe_3,-399,104.0,583,14.89
fe_4,-307,106.0,562,14.46
fe_5,-343,107.0,570,14.27
fe_6,-321,103.0,565,15.02
fe_7,-368,107.0,537,14.15
fe_8,-361,111.0,573,13.41
fe_9,-328,115.0,577,12.64
fe_10,-317,118.0,537,12.07


## Targeted Data Quality Checks

Four checks cover numeric validity, loan counts and types, categorical codes, and credit history.

- Use an inclusive age range of **0 ~ 150**.
- Check non-negative values where required by field meaning. Negative credit-limit changes, delay values, and balances are not universally invalid.
- Flag the observed extreme balance code for possible null conversion rather than deleting the customer.

In [ ]:
AGE_MIN, AGE_MAX = 0, 150
review_rules = [
    ("attributes", "Age", AGE_MIN, AGE_MAX),
    ("financials", "Annual_Income", 0, None),
    ("financials", "Monthly_Inhand_Salary", 0, None),
    ("financials", "Num_Bank_Accounts", 0, None),
    ("financials", "Num_Credit_Card", 0, None),
    ("financials", "Num_of_Loan", 0, None),
    ("financials", "Num_of_Delayed_Payment", 0, None),
    ("financials", "Num_Credit_Inquiries", 0, None),
    ("financials", "Outstanding_Debt", 0, None),
    ("financials", "Total_EMI_per_month", 0, None),
    ("financials", "Amount_invested_monthly", 0, None),
    ("financials", "Credit_Utilization_Ratio", 0, 100),
]
range_rows = []
for name, col, lower, upper in review_rules:
    values = numeric_views[name][col]
    invalid = values.lt(lower)
    if upper is not None:
        invalid = invalid | values.gt(upper)
    invalid = invalid.fillna(False)
    range_rows.append({
        "table": name, "column": col, "lower": lower, "upper": upper,
        "invalid_rows": int(invalid.sum()),
        "invalid_pct": round(invalid.mean() * 100, 2),
        "raw_examples": repr(tables[name].loc[invalid.fillna(False), col].head(3).tolist()),
    })
range_report = pd.DataFrame(range_rows)
print("7A. Range checks: null upper means no cap; missing values are excluded")
display(range_report)

# Flag the observed sentinel, not all negative balances.
BALANCE_SENTINEL = "__-333333333333333333333333333__"
balance_flag = tables["financials"]["Monthly_Balance"].str.strip().eq(BALANCE_SENTINEL)
balance_review = pd.DataFrame({
    "raw_value": tables["financials"].loc[balance_flag, "Monthly_Balance"],
    "numeric_preview": numeric_views["financials"].loc[balance_flag, "Monthly_Balance"],
})
print("Monthly_Balance sentinel count:", int(balance_flag.sum()))
display(balance_review)

7A. Range checks: null upper means no cap; missing values are excluded


,table,column,lower,upper,invalid_rows,invalid_pct,raw_examples
0,attributes,Age,0,150.0,319,2.55,"['4746', '3843_', '5498']"
1,financials,Annual_Income,0,NaN,0,0.00,[]
2,financials,Monthly_Inhand_Salary,0,NaN,0,0.00,[]
3,financials,Num_Bank_Accounts,0,NaN,4,0.03,"['-1', '-1', '-1']"
4,financials,Num_Credit_Card,0,NaN,0,0.00,[]
5,financials,Num_of_Loan,0,NaN,500,4.00,"['-100', '-100', '-100']"
6,financials,Num_of_Delayed_Payment,0,NaN,84,0.67,"['-2', '-1', '-1']"
7,financials,Num_Credit_Inquiries,0,NaN,0,0.00,[]
8,financials,Outstanding_Debt,0,NaN,0,0.00,[]
9,financials,Total_EMI_per_month,0,NaN,0,0.00,[]


Monthly_Balance sentinel count: 1


,raw_value,numeric_preview
2003,__-333333333333333333333333333__,-333333333333333314856026112.0


In [ ]:
print("7B. Loan count and type consistency")
financials = tables["financials"]
loan_count = numeric_views["financials"]["Num_of_Loan"]
loan_type = financials["Type_of_Loan"].str.strip()

# Financial loan counts need not equal the number of loans in this LMS dataset.
count_status = pd.Series("missing_or_unparseable", index=financials.index)
count_status.loc[loan_count.eq(0)] = "zero"
count_status.loc[loan_count.gt(0)] = "positive"
invalid_count = loan_count.lt(0) | (loan_count.notna() & loan_count.mod(1).ne(0))
count_status.loc[invalid_count.fillna(False)] = "invalid_count"

has_unspecified = loan_type.str.contains("Not Specified", case=False, regex=False)
type_status = pd.Series("specified", index=financials.index)
type_status.loc[has_unspecified] = "partly_unspecified"
type_status.loc[loan_type.str.lower().eq("not specified")] = "unspecified_only"
type_status.loc[missing_candidates(loan_type)] = "other_missing"
type_status.loc[loan_type.eq("")] = "blank"

display(pd.crosstab(count_status, type_status, margins=True).rename_axis(
    index="loan_count_status", columns="loan_type_status"
))

blank_type = loan_type.eq("")
no_loan = count_status.eq("zero") & blank_type
missing_type = count_status.eq("positive") & type_status.isin(["blank", "other_missing", "unspecified_only"])
partial_type = count_status.eq("positive") & type_status.eq("partly_unspecified")
zero_but_has_type = count_status.eq("zero") & type_status.isin(["specified", "partly_unspecified"])
blank_with_bad_count = blank_type & count_status.isin(["invalid_count", "missing_or_unparseable"])

# Only a blank type with zero count supports No_Loan; other unknowns remain unresolved.
loan_type_checks = pd.Series({
    "zero_count_and_blank__No_Loan_candidate": int(no_loan.sum()),
    "positive_count_but_missing_or_unspecified_type": int(missing_type.sum()),
    "positive_count_with_partly_unspecified_type": int(partial_type.sum()),
    "zero_count_but_has_type__conflict": int(zero_but_has_type.sum()),
    "blank_type_with_invalid_or_missing_count__unresolved": int(blank_with_bad_count.sum()),
    "invalid_loan_count_total": int(count_status.eq("invalid_count").sum()),
}, name="rows")
display(loan_type_checks.to_frame())

review = blank_with_bad_count | missing_type | zero_but_has_type
print("Review samples: loan counts and types, up to 10 rows")
display(financials.loc[review, ["Num_of_Loan", "Type_of_Loan"]].head(10))

7B. Loan count and type consistency


loan_type_status,blank,partly_unspecified,specified,unspecified_only,All
loan_count_status,,,,,
invalid_count,53,160,279,8,500
positive,10,3624,6835,168,10637
zero,1363,0,0,0,1363
All,1426,3784,7114,176,12500


,rows
zero_count_and_blank__No_Loan_candidate,1363
positive_count_but_missing_or_unspecified_type,178
positive_count_with_partly_unspecified_type,3624
zero_count_but_has_type__conflict,0
blank_type_with_invalid_or_missing_count__unresolved,53
invalid_loan_count_total,500


Review samples: loan counts and types, up to 10 rows


,Num_of_Loan,Type_of_Loan
182,1,Not Specified
229,1,Not Specified
434,1,Not Specified
456,-100,
470,1,Not Specified
554,1,Not Specified
555,1,Not Specified
604,1,Not Specified
670,1,Not Specified
732,-100,


In [ ]:
print("7C. Special codes and unknown categories")
category_fields = [
    ("attributes", "Occupation"),
    ("financials", "Credit_Mix"),
    ("financials", "Payment_of_Min_Amount"),
    ("financials", "Payment_Behaviour"),
]
for name, col in category_fields:
    counts = tables[name][col].str.strip().value_counts(dropna=False)
    print(col)
    display(pd.DataFrame({"rows": counts, "pct": (counts / len(tables[name]) * 100).round(2)}))

valid_behaviours = {
    "Low_spent_Small_value_payments", "Low_spent_Medium_value_payments",
    "Low_spent_Large_value_payments", "High_spent_Small_value_payments",
    "High_spent_Medium_value_payments", "High_spent_Large_value_payments",
}
behaviour = financials["Payment_Behaviour"].str.strip()
behaviour_missing = missing_candidates(behaviour)
behaviour_unexpected = ~behaviour.isin(valid_behaviours) & ~behaviour_missing
behaviour_review = pd.Series({
    "missing_candidates": int(behaviour_missing.sum()),
    "unexpected_code_rows": int(behaviour_unexpected.sum()),
    "missing_or_unexpected_pct": round((behaviour_missing | behaviour_unexpected).mean() * 100, 2),
}, name="value")
display(behaviour_review.to_frame())
display(behaviour[behaviour_unexpected].value_counts().to_frame("unexpected_rows"))
# Candidate unknowns: occupation and credit-mix placeholders and unexpected payment codes.
# Retain NM pending clarification; distinguish Unknown from No_Loan.

7C. Special codes and unknown categories
Occupation


,rows,pct
Occupation,,
_______,880,7.04
Lawyer,828,6.62
Architect,795,6.36
Engineer,793,6.34
Accountant,791,6.33
Scientist,789,6.31
Teacher,782,6.26
Mechanic,780,6.24
Media_Manager,780,6.24


Credit_Mix


,rows,pct
Credit_Mix,,
Standard,4497,35.98
Good,3032,24.26
_,2611,20.89
Bad,2360,18.88


Payment_of_Min_Amount


,rows,pct
Payment_of_Min_Amount,,
Yes,6571,52.57
No,4491,35.93
NM,1438,11.5


Payment_Behaviour


,rows,pct
Payment_Behaviour,,
Low_spent_Small_value_payments,3202,25.62
High_spent_Medium_value_payments,2242,17.94
Low_spent_Medium_value_payments,1686,13.49
High_spent_Large_value_payments,1683,13.46
High_spent_Small_value_payments,1389,11.11
Low_spent_Large_value_payments,1300,10.4
!@9#%8,998,7.98


,value
missing_candidates,0.00
unexpected_code_rows,998.00
missing_or_unexpected_pct,7.98


,unexpected_rows
Payment_Behaviour,
!@9#%8,998


In [ ]:
print("7D. Credit history in months: conversion preview")
history_raw = financials["Credit_History_Age"].str.strip()
history_parts = history_raw.str.extract(r"^(\d+) Years? and (\d+) Months?$")
years = pd.to_numeric(history_parts[0], errors="coerce")
months = pd.to_numeric(history_parts[1], errors="coerce")
history_months = years * 12 + months
history_report = pd.Series({
    "matched_format": int(history_months.notna().sum()),
    "missing_candidates": int(missing_candidates(history_raw).sum()),
    "unmatched_nonmissing": int((history_months.isna() & ~missing_candidates(history_raw)).sum()),
    "month_component_above_11": int(months.gt(11).sum()),
    "min_months": history_months.min(),
    "median_months": history_months.median(),
    "max_months": history_months.max(),
}, name="value")
display(history_report.to_frame())

7D. Credit history in months: conversion preview


,value
matched_format,12500.0
missing_candidates,0.0
unmatched_nonmissing,0.0
month_component_above_11,0.0
min_months,8.0
median_months,222.0
max_months,404.0


## Loan Lifecycle and Label Coverage

Check that customer, start date, and tenure remain consistent within each loan, and that installment numbers match elapsed calendar months.

Verify month 6 coverage for the Lab 2 label definition. Missing observations must not be interpreted as non-defaults. 

In [ ]:
loans = tables["loans"]
loan_numbers = numeric_views["loans"]
loan_start = pd.to_datetime(loans["loan_start_date"], format="%Y-%m-%d", errors="coerce")
loan_snapshot = pd.to_datetime(loans["snapshot_date"], format="%Y-%m-%d", errors="coerce")
month_offset = (loan_snapshot.dt.year - loan_start.dt.year) * 12 + loan_snapshot.dt.month - loan_start.dt.month

loan_consistency = loans.groupby("loan_id")[["Customer_ID", "loan_start_date", "tenure"]].nunique(dropna=False)
display((loan_consistency > 1).sum().to_frame("loan_ids_with_inconsistent_values"))
valid_month_pair = month_offset.notna() & loan_numbers["installment_num"].notna()
loan_checks = {
    "snapshots_before_loan_start": int((loan_snapshot < loan_start).sum()),
    "installment_month_offset_mismatch": int((valid_month_pair & month_offset.ne(loan_numbers["installment_num"])).sum()),
    "negative_installment_num": int(loan_numbers["installment_num"].lt(0).sum()),
    "installment_above_tenure": int(loan_numbers["installment_num"].gt(loan_numbers["tenure"]).sum()),
    "zero_due_amount_rows": int(loan_numbers["due_amt"].eq(0).sum()),
    "zero_due_outside_mob0": int((loan_numbers["due_amt"].eq(0) & loan_numbers["installment_num"].ne(0)).sum()),
    "mob0_with_nonzero_due": int((loan_numbers["installment_num"].eq(0) & loan_numbers["due_amt"].ne(0)).sum()),
    "positive_overdue_with_zero_due": int((loan_numbers["due_amt"].eq(0) & loan_numbers["overdue_amt"].gt(0)).sum()),
}
display(pd.Series(loan_checks, name="rows").to_frame())

lifecycle = pd.DataFrame({
    "loan_id": loans["loan_id"], "installment": loan_numbers["installment_num"],
    "is_mob0": loan_numbers["installment_num"].eq(0),
    "is_mob6": loan_numbers["installment_num"].eq(6),
}).groupby("loan_id").agg(
    snapshot_rows=("installment", "size"),
    min_installment=("installment", "min"),
    max_installment=("installment", "max"),
    mob0_rows=("is_mob0", "sum"),
    mob6_rows=("is_mob6", "sum"),
)
display(lifecycle.describe())
display(pd.Series({
    "unique_loans": len(lifecycle),
    "loans_with_mob0": int(lifecycle["mob0_rows"].gt(0).sum()),
    "loans_with_exactly_one_mob6": int(lifecycle["mob6_rows"].eq(1).sum()),
    "loans_without_mob6": int(lifecycle["mob6_rows"].eq(0).sum()),
    "loans_with_multiple_mob6_rows": int(lifecycle["mob6_rows"].gt(1).sum()),
}, name="count").to_frame())

# This spine is for EDA joins only; no loan rows are silently selected from conflicting mappings.
loan_spine = loans[["loan_id", "Customer_ID", "loan_start_date"]].drop_duplicates().copy()
if loan_spine["loan_id"].duplicated().any():
    raise ValueError("Conflicting customer or start date for a loan_id; resolve before temporal analysis.")
loan_spine["application_date_proxy"] = pd.to_datetime(loan_spine["loan_start_date"], format="%Y-%m-%d", errors="coerce")
mob6_counts = lifecycle["mob6_rows"]
cohort = loan_spine.assign(
    application_month=loan_spine["application_date_proxy"].dt.to_period("M"),
    has_mob6=loan_spine["loan_id"].map(mob6_counts).gt(0),
).groupby("application_month").agg(loans=("loan_id", "size"), with_mob6=("has_mob6", "sum"))
cohort["without_mob6"] = cohort["loans"] - cohort["with_mob6"]
display(cohort)

,loan_ids_with_inconsistent_values
Customer_ID,0
loan_start_date,0
tenure,0


,rows
snapshots_before_loan_start,0
installment_month_offset_mismatch,0
negative_installment_num,0
installment_above_tenure,0
zero_due_amount_rows,12500
zero_due_outside_mob0,0
mob0_with_nonzero_due,0
positive_overdue_with_zero_due,0


,snapshot_rows,min_installment,max_installment,mob0_rows,mob6_rows
count,12500.0,12500.0,12500.0,12500.0,12500.0
mean,11.0,0.0,10.0,1.0,1.0
std,0.0,0.0,0.0,0.0,0.0
min,11.0,0.0,10.0,1.0,1.0
25%,11.0,0.0,10.0,1.0,1.0
50%,11.0,0.0,10.0,1.0,1.0
75%,11.0,0.0,10.0,1.0,1.0
max,11.0,0.0,10.0,1.0,1.0


,count
unique_loans,12500
loans_with_mob0,12500
loans_with_exactly_one_mob6,12500
loans_without_mob6,0
loans_with_multiple_mob6_rows,0


,loans,with_mob6,without_mob6
application_month,,,
2023-01,530,530,0
2023-02,501,501,0
2023-03,506,506,0
2023-04,510,510,0
2023-05,521,521,0
2023-06,517,517,0
2023-07,471,471,0
2023-08,481,481,0
2023-09,454,454,0


## Cross-Source Key Coverage

Compare customer-date pairs across feature sources. Deduplication is used only for key-set comparison, it does not clean the source tables.

In [ ]:
pair_rows = []
for left_name, right_name in [
    ("attributes", "financials"), ("attributes", "clickstream"), ("financials", "clickstream")
]:
    keys = ["Customer_ID", "snapshot_date"]
    left_keys = tables[left_name][keys].drop_duplicates()
    right_keys = tables[right_name][keys].drop_duplicates()
    comparison = left_keys.merge(right_keys, on=keys, how="outer", indicator=True, validate="one_to_one")
    counts = comparison["_merge"].value_counts()
    pair_rows.append({
        "left_table": left_name, "right_table": right_name,
        "same_customer_and_date": int(counts.get("both", 0)),
        "left_only_keys": int(counts.get("left_only", 0)),
        "right_only_keys": int(counts.get("right_only", 0)),
    })
display(pd.DataFrame(pair_rows))

,left_table,right_table,same_customer_and_date,left_only_keys,right_only_keys
0,attributes,financials,12500,0,0
1,attributes,clickstream,8974,3526,206402
2,financials,clickstream,8974,3526,206402


## Feature Timing Relative to Loan Start

Use `loan_start_date` as a proxy for application date. Count feature records before, on, and after that date. Future records are inspected only to identify leakage risks.

Distinguish customers absent from a source from those with only later records. Report coverage by loan cohort.

In [ ]:
temporal_rows = []
temporal_by_source = {}
for name in ["attributes", "financials", "clickstream"]:
    source = tables[name][["Customer_ID", "snapshot_date"]].copy()
    source["feature_date"] = pd.to_datetime(source["snapshot_date"], format="%Y-%m-%d", errors="coerce")
    # Multiple loans/customer and multiple feature dates/customer can both exist.
    # Keep loan_id in the joined result so coverage is computed per loan.
    joined = loan_spine[["loan_id", "Customer_ID", "application_date_proxy"]].merge(
        source[["Customer_ID", "feature_date"]], on="Customer_ID",
        how="left", indicator=True, validate="many_to_many"
    )
    joined["source_record"] = joined["_merge"].eq("both")
    joined["invalid_feature_date"] = joined["source_record"] & joined["feature_date"].isna()
    joined["before"] = joined["feature_date"].lt(joined["application_date_proxy"])
    joined["same_day"] = joined["feature_date"].eq(joined["application_date_proxy"])
    joined["after"] = joined["feature_date"].gt(joined["application_date_proxy"])
    per_loan = joined.groupby("loan_id")[["source_record", "invalid_feature_date", "before", "same_day", "after"]].sum()
    per_loan["on_or_before"] = per_loan["before"] + per_loan["same_day"]
    temporal_by_source[name] = per_loan
    total = len(per_loan)
    temporal_rows.append({
        "source": name, "total_loans": total,
        "no_customer_record": int(per_loan["source_record"].eq(0).sum()),
        "has_before": int(per_loan["before"].gt(0).sum()),
        "has_same_day": int(per_loan["same_day"].gt(0).sum()),
        "has_after": int(per_loan["after"].gt(0).sum()),
        "no_record_on_or_before": int(per_loan["on_or_before"].eq(0).sum()),
        "has_after_but_none_on_or_before": int((per_loan["after"].gt(0) & per_loan["on_or_before"].eq(0)).sum()),
        "on_or_before_coverage_pct": round(per_loan["on_or_before"].gt(0).mean() * 100, 2),
    })

temporal_report = pd.DataFrame(temporal_rows)
display(temporal_report)
coverage_by_month = loan_spine[["loan_id", "application_date_proxy"]].copy()
coverage_by_month["application_month"] = coverage_by_month["application_date_proxy"].dt.to_period("M")
for name, per_loan in temporal_by_source.items():
    coverage_by_month[name + "_on_or_before"] = coverage_by_month["loan_id"].map(per_loan["on_or_before"]).gt(0)
coverage_columns = [c for c in coverage_by_month if c.endswith("_on_or_before")]
print("Feature coverage on or before loan start, by cohort (%)")
display((coverage_by_month.groupby("application_month")[coverage_columns].mean() * 100).round(2))

print("Clickstream records per loan; no lookback window applied:")
display(temporal_by_source["clickstream"][["before", "same_day", "after"]].describe())

,source,total_loans,no_customer_record,has_before,has_same_day,has_after,no_record_on_or_before,has_after_but_none_on_or_before,on_or_before_coverage_pct
0,attributes,12500,0,0,12500,0,0,0,100.00
1,financials,12500,0,0,12500,0,0,0,100.00
2,clickstream,12500,3526,8444,8974,8974,3526,0,71.79


Feature coverage on or before loan start, by cohort (%)


,attributes_on_or_before,financials_on_or_before,clickstream_on_or_before
application_month,,,
2023-01,100.0,100.0,100.0
2023-02,100.0,100.0,100.0
2023-03,100.0,100.0,100.0
2023-04,100.0,100.0,100.0
2023-05,100.0,100.0,100.0
2023-06,100.0,100.0,100.0
2023-07,100.0,100.0,100.0
2023-08,100.0,100.0,100.0
2023-09,100.0,100.0,100.0


Clickstream records per loan; no lookback window applied:


,before,same_day,after
count,12500.000000,12500.00000,12500.000000
mean,6.074720,0.71792,10.437440
std,5.844727,0.45003,7.903653
min,0.000000,0.00000,0.000000
25%,0.000000,0.00000,0.000000
50%,5.000000,1.00000,11.000000
75%,11.000000,1.00000,17.000000
max,17.000000,1.00000,23.000000


## Key Findings

Summarize the main quality and coverage issues.

In [ ]:
follow_up = pd.Series({
    "age_outside_0_150": int(range_report.loc[range_report["column"].eq("Age"), "invalid_rows"].iloc[0]),
    "numeric_values_recoverable_by_edge_underscore_removal": int(numeric_report["recovered_by_edge_underscore_removal"].sum()),
    "monthly_balance_sentinel": int(balance_flag.sum()),
    "blank_type_and_zero_count__No_Loan_candidate": int(no_loan.sum()),
    "blank_type_with_invalid_count__needs_review": int(blank_with_bad_count.sum()),
    "unexpected_payment_behaviour": int(behaviour_unexpected.sum()),
    "loans_without_any_clickstream": int(temporal_by_source["clickstream"]["source_record"].eq(0).sum()),
}, name="count")
display(follow_up.to_frame())

,count
age_outside_0_150,319
numeric_values_recoverable_by_edge_underscore_removal,3191
monthly_balance_sentinel,1
blank_type_and_zero_count__No_Loan_candidate,1363
blank_type_with_invalid_count__needs_review,53
unexpected_payment_behaviour,998
loans_without_any_clickstream,3526
